# Facility Water Treatment Energy Estimation

Minimal implementation: definitions (with comments), training, prediction.

In [ ]:
# ---------------------------------------------------------------------------
# Shared numeric/pandas imports and one date helper
# ---------------------------------------------------------------------------
import numpy as np  # numeric cleanup, infinities, probability arrays
import pandas as pd  # CSV loading, cleaning, feature engineering


def _dates(values):
    # Parse ISO dates first, then day-first dates; unreadable text becomes NaT.
    s = values.astype('string')  # string dtype keeps missing values as <NA>
    parsed = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d %H:%M')
    missing = parsed.isna()  # ISO format could not read these rows
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(s[missing], errors='coerce', dayfirst=True)
    return parsed


# Only the imports this task actually uses.
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------------------------
# Feature contract for the numeric-only regression task
# ---------------------------------------------------------------------------
RAW_NUMERIC = ['flow_m3', 'occupancy', 'pump_hours', 'ambient_temp_c',
               'chemical_litres', 'pressure_bar']  # 6 sensor readings
DERIVED_NUMERIC = ['reading_hour', 'reading_month', 'is_weekend', 'flow_per_pump_hour']
FEATURES = RAW_NUMERIC + DERIVED_NUMERIC  # exactly 10 predictors
TARGET = 'energy_kwh'  # label, never used as a predictor


def load_data(filename):
    # A1: clean sensor readings and derive time / flow-rate features.
    df = pd.read_csv(filename).drop_duplicates('reading_id', keep='first').copy()
    df['reading_ts'] = _dates(df['reading_ts'])
    df = df[df['reading_ts'].notna()].copy()  # invalid dates cannot create time features
    for c in RAW_NUMERIC:
        values = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        df[c] = values.mask(values < 0)  # negative sensors are invalid and break log1p
    df['reading_hour'] = df['reading_ts'].dt.hour  # time-of-day effect on energy
    df['reading_month'] = df['reading_ts'].dt.month  # seasonal effect
    df['is_weekend'] = (df['reading_ts'].dt.day_of_week >= 5).astype(int)  # Sat/Sun flag
    # Divide only where pump_hours > 0, so zero pumps give NaN instead of inf.
    df['flow_per_pump_hour'] = df['flow_m3'] / df['pump_hours'].where(df['pump_hours'] > 0)
    return df


def build_pipeline(algorithm):
    # A2: fixed four-step numeric pipeline; log1p needs non-negative inputs.
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # fill missing readings
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),  # stabilise skew
        ('scaler', StandardScaler()),  # put features on comparable scales
        ('model', algorithm),  # supplied regressor
    ])


def prepare_datasets(train_file, test_file):
    # 1: filter invalid targets, keep aligned X/y pairs for train and test.
    result = []
    for filename in (train_file, test_file):
        df = load_data(filename)
        target = pd.to_numeric(df[TARGET], errors='coerce').replace([np.inf, -np.inf], np.nan)
        df = df[target.notna() & (target >= 0)].copy()  # negative/missing energy is unusable
        result.append((df[FEATURES], target.loc[df.index].astype(float)))
    return result[0][0], result[1][0], result[0][1], result[1][1]


def train_models(X_train, y_train):
    # 2: fit two different regressors with their own preprocessing copies.
    models = {
        'ridge_regression': Ridge(alpha=1.0),  # linear baseline
        'random_forest': RandomForestRegressor(n_estimators=80, max_depth=10, min_samples_leaf=2,
                                               random_state=42, n_jobs=-1),  # nonlinear model
    }
    for name, algo in models.items():
        models[name] = build_pipeline(algo).fit(X_train, y_train)
    return models


def evaluate_models(trained_models, X_test, y_test):
    # 3: compute three error metrics on held-back readings; lower MAE is better.
    rows = []
    for name, model in trained_models.items():
        pred = model.predict(X_test)
        rows.append({'model_name': name,
                     'mae': mean_absolute_error(y_test, pred),
                     'rmse': np.sqrt(mean_squared_error(y_test, pred)),
                     'r2': r2_score(y_test, pred)})
    return (pd.DataFrame(rows)
            .sort_values(['mae', 'rmse', 'model_name'], ascending=[True, True, True])
            .reset_index(drop=True))  # tie-breakers: RMSE, then name


def select_best_model(comparison, trained_models):
    # 4: first row is the best; hand back the original fitted pipeline.
    name = comparison.iloc[0]['model_name']
    return name, trained_models[name]


def predict_energy(model, df):
    # 5: predict on new readings, clip impossible negatives, round for reporting.
    values = np.round(np.clip(model.predict(df[FEATURES]), 0, None), 2)
    return pd.DataFrame({'reading_id': df['reading_id'].to_numpy(), 'predicted_energy_kwh': values})


def train():
    # TRAINING WORKFLOW: fit, rank, save the winner and the report.
    X_train, X_test, y_train, y_test = prepare_datasets('water_train.csv', 'water_test.csv')
    models = train_models(X_train, y_train)
    comparison = evaluate_models(models, X_test, y_test)
    name, best = select_best_model(comparison, models)
    joblib.dump(best, 'best_model.pkl')  # whole pipeline so prediction needs no setup
    comparison.to_csv('model_comparison.csv', index=False)
    return name, comparison


def predict():
    # PREDICTION WORKFLOW: load saved pipeline, produce predictions only.
    model = joblib.load('best_model.pkl')
    result = predict_energy(model, load_data('water_predict.csv'))
    result.to_csv('water_predictions.csv', index=False)
    return result


In [ ]:
# TRAINING WORKFLOW: this is the only cell that fits models and writes artifacts.
name, comparison = train()
comparison

In [ ]:
# PREDICTION WORKFLOW: reload the saved model; no fitting happens here.
predict()